# 06 — Final model comparison

Selects the local-UI model using validation-seen IoU only.


In [ ]:
from pathlib import Path
import json
import os
import sys

from IPython.display import Image, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "datasets").is_dir() and (path / "final_model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
FRESH_TRAINING = True
RESULT_ROOT = PROJECT_ROOT / "training_results"
POINT_ROOT = RESULT_ROOT
print("Project:", PROJECT_ROOT)
print("Run ID:", RUN_ID)


## Comparison and selection implementation


In [ ]:
import json
import os
import shutil

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

EXPERIMENTS = {'baseline_object_mask': None, 'fixed_uvd': None, 'query_gated_uvd': None, 'rotation_consistent': None, 'geometry_dropout': None}

def run_id():
    return os.environ.get("FINAL_TRAINING_RUN_ID", "manual")

def results_root():
    path = PROJECT_ROOT / "training_results"
    path.mkdir(parents=True, exist_ok=True)
    return path

def points_root():
    path = PROJECT_ROOT / "training_results"
    path.mkdir(parents=True, exist_ok=True)
    return path

def build_comparison() -> pd.DataFrame:
    result_base = results_root()
    point_base = points_root()
    missing = [name for name in EXPERIMENTS if not (result_base / name / "summary.csv").is_file()]
    if missing:
        raise FileNotFoundError(f"Training summaries are missing for: {missing}")
    frames = [pd.read_csv(result_base / name / "summary.csv") for name in EXPERIMENTS]
    comparison = pd.concat(frames, ignore_index=True)
    comparison.to_csv(result_base / "all_experiment_comparison.csv", index=False)
    validation = comparison.drop_duplicates("experiment").sort_values("validation_iou", ascending=False)
    selected = str(validation.iloc[0].experiment)
    shutil.copy2(point_base / selected / "ui_model.pt", point_base / "best_model.pt")
    registry = {
        "run_id": run_id(),
        "selection_rule": "maximum validation_seen IoU; no test metric used for selection",
        "selected_model": selected,
        "selected_checkpoint": str((point_base / "best_model.pt").relative_to(PROJECT_ROOT)),
        "models": {
            name: {
                "checkpoint": str((point_base / name / "ui_model.pt").relative_to(PROJECT_ROOT)),
                "validation_iou": float(validation.set_index("experiment").loc[name, "validation_iou"]),
                "selected_epoch": int(validation.set_index("experiment").loc[name, "selected_epoch"]),
            }
            for name in EXPERIMENTS
        },
    }
    (point_base / "model_registry.json").write_text(json.dumps(registry, indent=2) + "\n")
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    validation.sort_values("validation_iou").plot.barh(
        x="experiment", y="validation_iou", legend=False, ax=axes[0], title="Validation checkpoint selection"
    )
    test = comparison.pivot(index="experiment", columns="split", values="iou")
    test.plot.bar(ax=axes[1], title="Final seen and unseen test IoU")
    axes[0].set_xlabel("Validation IoU")
    axes[1].set_ylabel("Mean IoU")
    axes[1].tick_params(axis="x", rotation=30)
    figure.tight_layout()
    figure.savefig(result_base / "final_model_comparison.png", dpi=180, bbox_inches="tight")
    plt.close(figure)
    print(comparison.to_string(index=False))
    print("Selected model:", selected)
    print("Selected UI checkpoint:", point_base / "best_model.pt")
    return comparison


## Run comparison


In [ ]:
comparison = build_comparison()
display(comparison.round(4))
display(Image(filename=str(RESULT_ROOT / "final_model_comparison.png")))
registry = json.loads((POINT_ROOT / "model_registry.json").read_text())
display(pd.DataFrame(registry["models"]).T.sort_values("validation_iou", ascending=False).round(4))
print("Selected model:", registry["selected_model"])
print("Selected UI checkpoint:", POINT_ROOT / "best_model.pt")
